# Phase 6 - Probabilistic forecasting (quantile LightGBM)

Three independent LightGBM models, one per quantile (alpha = 0.10 / 0.50 / 0.90). Same supervised feature table and same backtest origins as Phase 5; the difference is the loss.

What this notebook does:

1. Run `train_quantile_lightgbm.run()`.
2. Inspect overall p10-p90 coverage. Target is around 80%; below 75% the intervals are too narrow, above 90% too wide.
3. Coverage / pinball / interval width broken out by horizon and category.
4. Plot a p10-p50-p90 fan against actuals for one example series.
5. Compare p50 against the Phase 5 point model on MAE / WAPE / Bias.

All real logic lives in `seercast.models.quantile_lightgbm`, `seercast.evaluation.probabilistic_metrics`, and `seercast.training.train_quantile_lightgbm`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from seercast.training.train_quantile_lightgbm import run as run_quantile
from seercast.evaluation.probabilistic_metrics import all_quantile_metrics

REPO_ROOT

## 1. Run the quantile backtest

In [ ]:
result = run_quantile()
predictions = result['predictions']
scores = result['scores']
diagnostics = result['diagnostics']
p50_vs_point = result['p50_vs_point']
models = result['models']
predictions.head()

## 2. Overall metrics + coverage interpretation

In [ ]:
overall = all_quantile_metrics(predictions)
for k, v in overall.items():
    print(f'{k:40s} {v:.4f}')

cov = overall['coverage_p10_p90']
if cov < 0.75:
    print(f'\nCoverage {cov:.1%} is below 75% - intervals are likely too narrow (overconfident).')
elif cov > 0.90:
    print(f'\nCoverage {cov:.1%} is above 90% - intervals are likely too wide (over-conservative).')
else:
    print(f'\nCoverage {cov:.1%} is in the healthy 75-90% band (target around 80%).')

## 3. Coverage / pinball by horizon

Shorter horizons should generally have tighter intervals and higher pinball loss is bad.

In [ ]:
by_h = diagnostics[diagnostics['dimension'] == 'horizon'].copy()
by_h['horizon'] = by_h['value'].astype(int)
by_h = by_h.sort_values('horizon')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.plot(by_h['horizon'], by_h['coverage_p10_p90'], marker='o', label='coverage')
ax.axhline(0.80, color='gray', linestyle='--', label='target 0.80')
ax.set_xlabel('horizon')
ax.set_ylabel('p10-p90 coverage')
ax.set_title('Coverage by horizon')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(by_h['horizon'], by_h['interval_width_p10_p90'], marker='o', label='width')
ax.set_xlabel('horizon')
ax.set_ylabel('mean(p90 - p10)')
ax.set_title('Interval width by horizon')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Coverage by category

In [ ]:
diagnostics[diagnostics['dimension'] == 'category']

## 5. p10-p90 fan vs actuals for one example series

In [ ]:
latest = predictions['origin_date'].max()
top_id = (
    predictions[predictions['origin_date'] == latest]
    .groupby('id')['actual'].sum().idxmax()
)
sub = predictions[(predictions['origin_date'] == latest) & (predictions['id'] == top_id)].sort_values('horizon')

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(sub['target_date'], sub['p10'], sub['p90'], alpha=0.25, label='p10-p90')
ax.plot(sub['target_date'], sub['p50'], marker='o', label='p50')
ax.plot(sub['target_date'], sub['actual'], color='black', marker='.', label='actual')
ax.set_title(f'{top_id} - p10/p50/p90 fan (origin {latest.date()})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. p50 vs Phase 5 point model

On the same backtest origins. The two are independent fits with different objectives (p50 minimizes pinball at 0.5 = MAE; the point model minimizes Poisson deviance), so neither is guaranteed to win.

In [ ]:
p50_vs_point

**Next:** Phase 7 - predictive scenario simulation (price / event / SNAP / momentum what-ifs).